# Objectives

### English

- Explore the international competitions available in the StatsBomb Open Data repository.
- Select recent international men's competitions suitable for the V1 player database.
- Generalize the player vector generation pipeline beyond FIFA World Cup data.
- Build a unified international player database by aggregating player event vectors across multiple competitions.
- Validate the consistency of player identities and save the resulting datasets for future stages of the project.

### Español

- Explorar las competiciones internacionales disponibles en el repositorio StatsBomb Open Data.
- Seleccionar las competiciones internacionales masculinas recientes adecuadas para la base de jugadores V1.
- Generalizar el pipeline de generación de vectores de jugadores más allá de los datos de la Copa del Mundo.
- Construir una base de datos unificada de jugadores internacionales agregando los vectores de eventos de múltiples competiciones.
- Validar la consistencia de la identidad de los jugadores y guardar los datasets resultantes para las siguientes etapas del proyecto.

# Build Structure

## Imports

In [161]:
import json
import pandas as pd
from tqdm import tqdm
from pathlib import Path


## Utility Functions

Updated from Notebook 02

In [162]:
def build_player_vector(events, expected_teams=2, verbose=False):
    player_events = []

    for event in events:
        if "player" not in event:
            continue

        player_events.append({
            "player_id": event["player"]["id"],
            "player_name": event["player"]["name"],
            "team_name": event["team"]["name"],
            "event_type": event["type"]["name"]
        })

    player_events_df = pd.DataFrame(player_events)

    ADMIN_EVENTS = [
        "Starting XI",
        "Half Start",
        "Half End",
        "Substitution",
        "Tactical Shift",
        "Injury Stoppage",
        "Referee Ball-Drop",
    ]

    player_events_clean = player_events_df[
        ~player_events_df["event_type"].isin(ADMIN_EVENTS)
    ].copy()

    player_vector = (
        player_events_clean
        .pivot_table(
            index=["player_id", "player_name", "team_name"],
            columns="event_type",
            values="event_type",
            aggfunc="count",
            fill_value=0
        )
        .reset_index()
    )

    assert player_vector["player_id"].is_unique, "Duplicated players in match-level vector"
    assert player_vector.isna().sum().sum() == 0, "NaN values found"
    assert "Pass" in player_vector.columns, "Pass column not found"

    if expected_teams is not None:
        assert player_vector["team_name"].nunique() == expected_teams, "Unexpected number of teams"

    if verbose:
        print("Player vector created successfully")

    return player_vector



In [163]:
def build_worldcup_player_vector(matches_df, events_dir, save_path=None):
    """
    Construye un player vector acumulado para todos los partidos de un Mundial.

    Entrada:
    - matches_df: DataFrame con columna match_id
    - events_dir: carpeta donde están los events JSON
    - save_path: ruta opcional para guardar CSV

    Salida:
    - player_vector_worldcup: 1 fila por jugador en el torneo
    - player_vectors_match_level: 1 fila por jugador por partido
    """

    all_player_vectors = []

    for match_id in matches_df["match_id"]:

        event_file = events_dir / f"{match_id}.json"

        with open(event_file, "r", encoding="utf-8") as f:
            events = json.load(f)

        player_vector = build_player_vector(events)

        player_vector["match_id"] = match_id

        all_player_vectors.append(player_vector)

    # 1 fila por jugador por partido
    player_vectors_match_level = pd.concat(
    all_player_vectors,
    ignore_index=True
)

    player_vectors_match_level = player_vectors_match_level.fillna(0)

    # Columnas identificatorias
    id_cols = [
        "player_id",
        "player_name",
        "team_name"
    ]

    # Columnas numéricas/eventos
    numeric_cols = [
    col for col in player_vectors_match_level.select_dtypes(include="number").columns
    if col not in ["match_id", "player_id"]
    ]

    # 1 fila por jugador en todo el Mundial
    player_vector_worldcup = (
        player_vectors_match_level
        .groupby(id_cols)[numeric_cols]
        .sum()
        .reset_index()
    )

    # Tests básicos
    assert player_vector_worldcup["player_id"].is_unique, "Hay jugadores duplicados"
    assert player_vectors_match_level.isna().sum().sum() == 0
    assert player_vector_worldcup.isna().sum().sum() == 0, "Hay valores NaN"
    assert "Pass" in player_vector_worldcup.columns, "No existe la columna Pass"

    print("World Cup player vector created successfully")
    print("Match-level shape:", player_vectors_match_level.shape)
    print("World Cup-level shape:", player_vector_worldcup.shape)

    if save_path is not None:
        player_vector_worldcup.to_csv(save_path, index=False)
        print("Saved to:", save_path)

    return player_vector_worldcup, player_vectors_match_level

    

In [164]:
def build_competition_player_vectors(
    matches_df,
    events_dir,
    competition_name,
    season_name,
    competition_id,
    season_id,
):
    """
    Builds player vectors for all matches in a competition season.

    Returns:
    - player_vector_competition: one row per player in the competition
    - player_vectors_match_level: one row per player per match
    """

    all_player_vectors = []

    for match_id in tqdm(matches_df["match_id"], desc=f"{competition_name} {season_name}"):

        event_file = events_dir / f"{match_id}.json"

        with open(event_file, "r", encoding="utf-8") as f:
            events = json.load(f)

        player_vector = build_player_vector(events, expected_teams=2, verbose=False)

        player_vector["match_id"] = match_id
        player_vector["competition_id"] = competition_id
        player_vector["season_id"] = season_id
        player_vector["competition_name"] = competition_name
        player_vector["season_name"] = season_name

        all_player_vectors.append(player_vector)

    player_vectors_match_level = pd.concat(
    all_player_vectors,
    ignore_index=True
    )

    player_vectors_match_level = player_vectors_match_level.fillna(0)

    

    id_cols = [
        "player_id",
        "player_name",
        "team_name",
        "competition_id",
        "season_id",
        "competition_name",
        "season_name"
    ]

    metadata_cols = id_cols + ["match_id"]

    numeric_cols = [
        col for col in player_vectors_match_level.select_dtypes(include="number").columns
        if col not in metadata_cols
    ]

    player_vector_competition = (
        player_vectors_match_level
        .groupby(id_cols)[numeric_cols]
        .sum()
        .reset_index()
    )

    assert player_vector_competition.isna().sum().sum() == 0, "NaN values found"
    assert "Pass" in player_vector_competition.columns, "Pass column not found"
    assert player_vectors_match_level.isna().sum().sum() == 0, "NaN values found in match-level vectors"
    return player_vector_competition, player_vectors_match_level



## Paths

In [165]:
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

STATSBOMB_DIR = RAW_DIR / "statsbomb" / "data"

COMPETITIONS_PATH = STATSBOMB_DIR / "competitions.json"
MATCHES_DIR = STATSBOMB_DIR / "matches"
EVENTS_DIR = STATSBOMB_DIR / "events"

## Load Competitions

In [166]:
competitions = pd.read_json(COMPETITIONS_PATH)

competitions.head()

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-11-15T23:17:41.827093,2025-11-15T23:17:41.827093,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381
2,1267,107,Africa,African Cup of Nations,male,False,True,2023,2026-05-12T21:18:08.827431,2026-05-02T02:07:18.902396,2026-05-02T02:07:18.902396,2026-05-12T21:18:08.827431
3,16,4,Europe,Champions League,male,False,False,2018/2019,2026-05-15T15:54:04.598614,2021-06-13T16:17:31.694,NaN,2026-05-15T15:54:04.598614
4,16,1,Europe,Champions League,male,False,False,2017/2018,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,NaN,2024-02-13T02:35:28.134882


## Select candidate competitions

In [167]:
international_male = competitions[
    (competitions["competition_gender"] == "male") &
    (competitions["competition_international"] == True) &
    (competitions["competition_youth"] == False)
].copy()

candidate_keywords = [
    "FIFA World Cup",
    "Copa America",
    "UEFA Euro",
    "African Cup of Nations",
]

pattern = "|".join(candidate_keywords)

candidate_competitions = international_male[
    international_male["competition_name"].str.contains(
        pattern,
        case=False,
        na=False
    )
].copy()

candidate_competitions[
    [
        "competition_id",
        "season_id",
        "competition_name",
        "season_name",
        "country_name",
    ]
].sort_values(["competition_name", "season_name"])

,competition_id,season_id,competition_name,season_name,country_name
2,1267,107,African Cup of Nations,2023,Africa
21,223,282,Copa America,2024,South America
37,43,269,FIFA World Cup,1958,International
36,43,270,FIFA World Cup,1962,International
35,43,272,FIFA World Cup,1970,International
34,43,51,FIFA World Cup,1974,International
33,43,54,FIFA World Cup,1986,International
32,43,55,FIFA World Cup,1990,International
31,43,3,FIFA World Cup,2018,International
30,43,106,FIFA World Cup,2022,International


## Reduce to V1 Competitions

In [168]:
v1_competitions = candidate_competitions[
    (
        (candidate_competitions["competition_name"] == "FIFA World Cup") &
        (candidate_competitions["season_name"].isin(["2018", "2022"]))
    )
    |
    (
        (candidate_competitions["competition_name"] == "UEFA Euro") &
        (candidate_competitions["season_name"].isin(["2020", "2024"]))
    )
    |
    (
        (candidate_competitions["competition_name"] == "Copa America") &
        (candidate_competitions["season_name"].isin(["2024"]))
    )
    |
    (
        (candidate_competitions["competition_name"] == "African Cup of Nations") &
        (candidate_competitions["season_name"].isin(["2023"]))
    )
].copy()

v1_competitions[
    [
        "competition_id",
        "season_id",
        "competition_name",
        "season_name",
        "country_name",
    ]
].sort_values(["competition_name", "season_name"])

,competition_id,season_id,competition_name,season_name,country_name
2,1267,107,African Cup of Nations,2023,Africa
21,223,282,Copa America,2024,South America
31,43,3,FIFA World Cup,2018,International
30,43,106,FIFA World Cup,2022,International
74,55,43,UEFA Euro,2020,Europe
73,55,282,UEFA Euro,2024,Europe


## Test with one competition

In [169]:
test_competition = v1_competitions[
    (v1_competitions["competition_name"] == "Copa America") &
    (v1_competitions["season_name"] == "2024")
].iloc[0]

test_competition

competition_id                                      223
season_id                                           282
country_name                              South America
competition_name                           Copa America
competition_gender                                 male
competition_youth                                 False
competition_international                          True
season_name                                        2024
match_updated                2026-05-13T12:24:48.807600
match_updated_360                                   NaN
match_available_360                                 NaN
match_available              2026-05-13T12:24:48.807600
Name: 21, dtype: object

In [170]:
competition_id = test_competition["competition_id"]
season_id = test_competition["season_id"]
competition_name = test_competition["competition_name"]
season_name = test_competition["season_name"]

matches_path = MATCHES_DIR / str(competition_id) / f"{season_id}.json"

matches_df = pd.read_json(matches_path)

matches_df.shape

(32, 18)

In [171]:
matches_df.head()

,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,metadata,match_week,competition_stage,stadium,referee
0,3939991,2024-07-03,01:00:00.000,"{'competition_id': 223, 'country_name': 'South...","{'season_id': 282, 'season_name': '2024'}","{'home_team_id': 781, 'home_team_name': 'Brazi...","{'away_team_id': 769, 'away_team_name': 'Colom...",1,1,available,unscheduled,2026-05-02T02:02:52.324442,NaN,"{'data_version': '1.1.0', 'shot_fidelity_versi...",3,"{'id': 10, 'name': 'Group Stage'}","{'id': 1001237, 'name': 'Levi''s Stadium', 'co...","{'id': 1847, 'name': 'Jesús Valenzuela Sáez', ..."
1,3939987,2024-07-01,00:00:00.000,"{'competition_id': 223, 'country_name': 'South...","{'season_id': 282, 'season_name': '2024'}","{'home_team_id': 2305, 'home_team_name': 'Jama...","{'away_team_id': 3563, 'away_team_name': 'Vene...",0,3,available,unscheduled,2026-05-02T01:59:50.209451,NaN,"{'data_version': '1.1.0', 'shot_fidelity_versi...",3,"{'id': 10, 'name': 'Group Stage'}","{'id': 117996, 'name': 'Q2 Stadium', 'country'...","{'id': 252, 'name': 'Maurizio Mariani', 'count..."
2,3939982,2024-06-28,01:00:00.000,"{'competition_id': 223, 'country_name': 'South...","{'season_id': 282, 'season_name': '2024'}","{'home_team_id': 783, 'home_team_name': 'Urugu...","{'away_team_id': 3564, 'away_team_name': 'Boli...",5,0,available,unscheduled,2026-05-01T16:26:23.907380,NaN,"{'data_version': '1.1.0', 'shot_fidelity_versi...",2,"{'id': 10, 'name': 'Group Stage'}","{'id': 1001990, 'name': 'MetLife Stadium', 'co...","{'id': 1852, 'name': 'Juan Gabriel Benítez', '..."
3,3939970,2024-06-22,00:00:00.000,"{'competition_id': 223, 'country_name': 'South...","{'season_id': 282, 'season_name': '2024'}","{'home_team_id': 784, 'home_team_name': 'Peru'...","{'away_team_id': 3562, 'away_team_name': 'Chil...",0,0,available,unscheduled,2026-05-01T16:26:44.940945,NaN,"{'data_version': '1.1.0', 'shot_fidelity_versi...",1,"{'id': 10, 'name': 'Group Stage'}","{'id': 94424, 'name': 'AT&T Stadium', 'country...","{'id': 986, 'name': 'Wilton Pereira Sampaio', ..."
4,3939980,2024-06-26,22:00:00.000,"{'competition_id': 223, 'country_name': 'South...","{'season_id': 282, 'season_name': '2024'}","{'home_team_id': 3565, 'home_team_name': 'Ecua...","{'away_team_id': 2305, 'away_team_name': 'Jama...",3,1,available,unscheduled,2026-05-01T16:00:45.077382,NaN,"{'data_version': '1.1.0', 'shot_fidelity_versi...",2,"{'id': 10, 'name': 'Group Stage'}","{'id': 1000154, 'name': 'Allegiant Stadium', '...","{'id': 1773, 'name': 'Cristián Marcelo Garay R..."


In [172]:
player_vector_competition, player_vectors_match_level = build_competition_player_vectors(
    matches_df=matches_df,
    events_dir=EVENTS_DIR,
    competition_name=competition_name,
    season_name=season_name,
    competition_id=competition_id,
    season_id=season_id,
)

Copa America 2024: 100%|██████████| 32/32 [00:01<00:00, 17.53it/s]


In [173]:
player_vector_competition.shape

(339, 32)

In [174]:
player_vector_competition.head()

event_type,player_id,player_name,team_name,competition_id,season_id,competition_name,season_name,50/50,Bad Behaviour,Ball Receipt*,...,Interception,Miscontrol,Pass,Player Off,Player On,Pressure,Shot,Shield,Offside,Own Goal Against
0,2995,Ángel Fabián Di María Hernández,Argentina,223,282,Copa America,2024,0,0.0,208,...,0,4,174,0.0,0.0,56,6,0.0,0.0,0.0
1,3063,Danilo Luiz da Silva,Brazil,223,282,Copa America,2024,3,0.0,198,...,7,0,259,0.0,0.0,20,1,0.0,0.0,0.0
2,3090,Nicolás Hernán Otamendi,Argentina,223,282,Copa America,2024,0,0.0,84,...,1,0,100,0.0,0.0,9,3,0.0,0.0,0.0
3,3159,Gregory Leigh,Jamaica,223,282,Copa America,2024,2,0.0,43,...,5,2,67,0.0,0.0,16,0,1.0,0.0,0.0
4,3312,Demarai Gray,Jamaica,223,282,Copa America,2024,1,0.0,57,...,3,7,43,0.0,0.0,20,2,0.0,0.0,0.0


In [175]:
player_vectors_match_level.shape

(1011, 33)

In [176]:
player_vectors_match_level.head()

event_type,player_id,player_name,team_name,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,Block,Carry,Clearance,...,Pressure,Shot,match_id,competition_id,season_id,competition_name,season_name,Shield,Offside,Own Goal Against
0,3063,Danilo Luiz da Silva,Brazil,2,0.0,32,1,4,28,5,...,10,0,3939991,223,282,Copa America,2024,0.0,0.0,0.0
1,3494,Davinson Sánchez Mina,Colombia,0,0.0,32,6,1,32,3,...,4,0,3939991,223,282,Copa America,2024,0.0,0.0,0.0
2,4372,Marcos Aoás Corrêa,Brazil,0,0.0,41,4,2,41,6,...,7,0,3939991,223,282,Copa America,2024,0.0,0.0,0.0
3,5547,Alisson Ramsés Becker,Brazil,0,0.0,16,1,0,17,0,...,0,0,3939991,223,282,Copa America,2024,0.0,0.0,0.0
4,5678,Jefferson Andrés Lerma Solís,Colombia,0,1.0,39,4,4,30,0,...,7,1,3939991,223,282,Copa America,2024,0.0,0.0,0.0


# Build International Player Database


## Main Structure

In [177]:
all_competition_vectors = []
all_match_level_vectors = []

for _, competition in tqdm(
    v1_competitions.iterrows(),
    total=len(v1_competitions),
    desc="Processing competitions"
):

    competition_id = competition["competition_id"]
    season_id = competition["season_id"]
    competition_name = competition["competition_name"]
    season_name = competition["season_name"]

    matches_path = MATCHES_DIR / str(competition_id) / f"{season_id}.json"

    matches_df = pd.read_json(matches_path)

    tqdm.write(
    f"{competition_name} {season_name}: {len(matches_df)} matches"
    )

    competition_vector, match_vector = build_competition_player_vectors(
        matches_df=matches_df,
        events_dir=EVENTS_DIR,
        competition_name=competition_name,
        season_name=season_name,
        competition_id=competition_id,
        season_id=season_id,
    )

    all_competition_vectors.append(competition_vector)
    all_match_level_vectors.append(match_vector)

Processing competitions:   0%|          | 0/6 [00:00<?, ?it/s]

African Cup of Nations 2023: 52 matches


Processing competitions:  17%|█▋        | 1/6 [00:02<00:14,  2.86s/it]

Copa America 2024: 32 matches


Processing competitions:  33%|███▎      | 2/6 [00:04<00:09,  2.34s/it]

...

Processing competitions:  67%|██████▋   | 4/6 [00:13<00:07,  3.61s/it]

UEFA Euro 2024: 51 matches


Processing competitions:  83%|████████▎ | 5/6 [00:17<00:03,  3.69s/it]

UEFA Euro 2020: 51 matches


Processing competitions: 100%|██████████| 6/6 [00:20<00:00,  3.47s/it]


In [178]:
player_vectors_competition_level = pd.concat(
    all_competition_vectors,
    ignore_index=True
).fillna(0)

player_vectors_match_level = pd.concat(
    all_match_level_vectors,
    ignore_index=True
).fillna(0)


In [179]:
assert player_vectors_competition_level.isna().sum().sum() == 0
assert player_vectors_match_level.isna().sum().sum() == 0

In [180]:
player_vectors_competition_level.shape

(3126, 33)

In [181]:
player_vectors_match_level.shape

(9560, 34)

### Checks

In [182]:
player_vectors_competition_level.isna().sum().sum()

np.int64(0)

In [183]:
player_vectors_match_level.isna().sum().sum()

np.int64(0)

In [184]:
player_vectors_competition_level[
    ["competition_name", "season_name"]
].drop_duplicates().sort_values(["competition_name", "season_name"])

event_type,competition_name,season_name
0,African Cup of Nations,2023
512,Copa America,2024
1532,FIFA World Cup,2018
851,FIFA World Cup,2022
2633,UEFA Euro,2020
2138,UEFA Euro,2024


## Build the Database

In [185]:
id_cols = ["player_id"]

metadata_cols = [
    "player_id",
    "competition_id",
    "season_id",
    "match_id"
]

numeric_cols = [
    col
    for col in player_vectors_match_level.select_dtypes(include="number").columns
    if col not in metadata_cols
]

international_player_database = (
    player_vectors_match_level
    .groupby(id_cols)[numeric_cols]
    .sum()
    .reset_index()
)


### Player Metadata

In [186]:
player_metadata = (
    player_vectors_match_level
    .groupby("player_id")
    .agg(
        player_name=("player_name", "first"),
        team_name=("team_name", "first"),
        matches_played=("match_id", "nunique"),
        competitions_played=("competition_name", "nunique"),
        seasons_played=("season_name", "nunique")
    )
    .reset_index()
)

#### Merge

In [187]:
international_player_database = international_player_database.merge(
    player_metadata,
    on="player_id",
    how="left"
)

#### Order Metadata

In [188]:
metadata_order = [
    "player_id",
    "player_name",
    "team_name",
    "matches_played",
    "competitions_played",
    "seasons_played"
]

event_cols = [
    col for col in international_player_database.columns
    if col not in metadata_order
]

international_player_database = international_player_database[
    metadata_order + event_cols
]


#### Checks

In [189]:
international_player_database.columns

Index(['player_id', 'player_name', 'team_name', 'matches_played',
       'competitions_played', 'seasons_played', '50/50', 'Bad Behaviour',
       'Ball Receipt*', 'Ball Recovery', 'Block', 'Carry', 'Clearance',
       'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Foul Committed',
       'Foul Won', 'Goal Keeper', 'Interception', 'Miscontrol', 'Pass',
       'Player Off', 'Player On', 'Pressure', 'Shield', 'Shot', 'Error',
       'Offside', 'Own Goal Against', 'Camera On'],
      dtype='str')

In [190]:
international_player_database.shape

(2279, 32)

In [191]:
international_player_database.isna().sum().sum()

np.int64(0)

In [192]:
international_player_database.head()

,player_id,player_name,team_name,matches_played,competitions_played,seasons_played,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,...,Pass,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On
0,2941,Ismaïla Sarr,Senegal,11,2,3,3.0,0.0,378,31,...,213,1.0,1.0,183,0.0,20,0.0,1.0,0.0,0.0
1,2948,Nabil Fekir,France,6,1,1,0.0,0.0,59,9,...,40,0.0,0.0,32,0.0,4,0.0,0.0,0.0,0.0
2,2954,Youri Tielemans,Belgium,14,2,4,1.0,0.0,475,24,...,506,0.0,0.0,165,0.0,10,0.0,0.0,0.0,0.0
3,2956,Bertrand Isidore Traoré,Burkina Faso,4,1,1,0.0,0.0,98,7,...,69,1.0,1.0,13,0.0,11,0.0,0.0,0.0,0.0
4,2972,Marcus Thuram,France,10,2,3,1.0,0.0,193,14,...,104,0.0,0.0,73,1.0,14,0.0,0.0,0.0,0.0


In [193]:
international_player_database["player_id"].is_unique

True

In [194]:
duplicated_players = international_player_database[
    international_player_database["player_id"].duplicated(keep=False)
].sort_values("player_id")

duplicated_players[
    ["player_id", "player_name", "team_name", "matches_played", "competitions_played", "seasons_played"]
]

,player_id,player_name,team_name,matches_played,competitions_played,seasons_played


In [195]:
duplicated_players["player_id"].nunique(), duplicated_players.shape[0]

(0, 0)

In [196]:
international_player_database.sort_values(
    "matches_played",
    ascending=False
).head(20)

,player_id,player_name,team_name,matches_played,competitions_played,seasons_played,50/50,Bad Behaviour,Ball Receipt*,Ball Recovery,...,Pass,Player Off,Player On,Pressure,Shield,Shot,Error,Offside,Own Goal Against,Camera On
62,3244,John Stones,England,26,2,4,1.0,0.0,1773,68,...,1956,1.0,1.0,137,1.0,16,0.0,0.0,0.0,0.0
120,3468,Jordan Pickford,England,26,2,4,0.0,0.0,569,114,...,950,0.0,0.0,1,1.0,0,1.0,0.0,0.0,0.0
1043,10955,Harry Kane,England,25,2,4,0.0,0.0,972,69,...,513,2.0,2.0,335,0.0,62,0.0,0.0,0.0,0.0
411,5487,Antoine Griezmann,France,24,2,4,6.0,1.0,993,80,...,957,1.0,1.0,453,0.0,47,0.0,2.0,0.0,0.0
10,3009,Kylian Mbappé Lottin,France,23,2,4,2.0,3.0,1087,75,...,703,4.0,4.0,208,0.0,79,0.0,0.0,0.0,0.0
384,5460,Andrej Kramarić,Croatia,21,2,4,1.0,0.0,650,42,...,468,1.0,1.0,198,1.0,30,0.0,1.0,0.0,0.0
57,3205,Kyle Walker,England,21,2,4,0.0,1.0,1215,81,...,1524,2.0,2.0,166,4.0,0,0.0,0.0,0.0,0.0
387,5463,Luka Modrić,Croatia,21,2,4,5.0,1.0,1403,125,...,1592,0.0,0.0,361,2.0,27,0.0,0.0,0.0,0.0
398,5474,Ivan Perišić,Croatia,20,2,4,1.0,0.0,818,47,...,673,1.0,1.0,319,5.0,51,0.0,1.0,0.0,0.0
84,3308,Kieran Trippier,England,20,2,4,1.0,1.0,851,55,...,1134,4.0,4.0,266,1.0,7,0.0,0.0,0.0,0.0


In [197]:
international_player_database["team_name"].value_counts().sort_values(ascending=False)

team_name
Spain                52
Poland               48
Germany              47
Brazil               47
Costa Rica           46
                     ..
Equatorial Guinea    19
Iceland              18
Slovenia             18
Zambia               18
Tanzania             17
Name: count, Length: 76, dtype: int64

In [198]:
international_player_database[
    [
        "player_name",
        "team_name",
        "matches_played",
        "competitions_played",
        "seasons_played",
    ]
].sort_values(
    "matches_played",
    ascending=False
).head(20)

,player_name,team_name,matches_played,competitions_played,seasons_played
62,John Stones,England,26,2,4
120,Jordan Pickford,England,26,2,4
1043,Harry Kane,England,25,2,4
411,Antoine Griezmann,France,24,2,4
10,Kylian Mbappé Lottin,France,23,2,4
384,Andrej Kramarić,Croatia,21,2,4
57,Kyle Walker,England,21,2,4
387,Luka Modrić,Croatia,21,2,4
398,Ivan Perišić,Croatia,20,2,4
84,Kieran Trippier,England,20,2,4


## Check for One Player -> Multiple Teams cases

### Player National Team Consistency

Verify that each player appears for only one national team across all selected international competitions.

This check validates that the player database can safely use `(player_id, team_name)` as its identifier for the current project version.

In [199]:
player_country_check = (
    player_vectors_match_level
    .groupby("player_id")["team_name"]
    .nunique()
    .sort_values(ascending=False)
)

player_country_check.head(20)

player_id
2941    1
2948    1
2954    1
2956    1
2972    1
2974    1
2988    1
2989    1
2995    1
2999    1
3009    1
3013    1
3017    1
3026    1
3027    1
3034    1
3035    1
3037    1
3042    1
3043    1
Name: team_name, dtype: int64

# Save Database

In [200]:
international_player_database_path = (
    PROCESSED_DIR / "international_player_database_v01.csv"
)

match_level_path = (
    PROCESSED_DIR / "international_player_vectors_match_level_v01.csv"
)

international_player_database.to_csv(
    international_player_database_path,
    index=False
)

player_vectors_match_level.to_csv(
    match_level_path,
    index=False
)

print("Saved:", international_player_database_path)
print("Saved:", match_level_path)

Saved: ..\data\processed\international_player_database_v01.csv
Saved: ..\data\processed\international_player_vectors_match_level_v01.csv


# Conclusions

This notebook created the first unified international player database for the project.

The database integrates player event vectors from recent international competitions:

- FIFA World Cup 2018
- FIFA World Cup 2022
- UEFA Euro 2020
- UEFA Euro 2024
- Copa America 2024
- African Cup of Nations 2023

The final dataset contains one row per player-national team combination and aggregates all available event counts across the selected competitions.

A consistency check confirmed that each player appears for only one national team within the V1 dataset.

This database will be used in the next notebook to construct roster-level vectors for current national teams.

In [201]:
summary = pd.DataFrame({
    "Metric": [
        "Competitions",
        "National teams",
        "Unique players",
        "Match-level player vectors",
        "Event features"
    ],
    "Value": [
        v1_competitions.shape[0],
        international_player_database["team_name"].nunique(),
        international_player_database.shape[0],
        player_vectors_match_level.shape[0],
        len(event_cols)
    ]
})

summary

,Metric,Value
0,Competitions,6
1,National teams,76
2,Unique players,2279
3,Match-level player vectors,9560
4,Event features,26
